# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

The FlyRank **Content Refresh & Opportunity Lane** supports a practical editorial decision: which pages should a human team audit first when a large publishing portfolio shows signs of search traffic decay? With a budget of 50 or fewer manual audits per week, the goal is a ranked, explainable queue rather than automated content changes.

**Research question:** Can pre-cutoff Search Console signals produce a useful, out-of-domain risk ranking for 15-day impression decline?

In [1]:
from pathlib import Path
import json
import pandas as pd

outputs_dir = Path('../outputs')
figures_dir = Path('../figures')
metrics_path = outputs_dir / 'model_metrics.json'
queue_path = outputs_dir / 'action_playbook_queue.csv'
figure_path = figures_dir / 'playbook_action_distribution.png'

assert metrics_path.exists()
assert queue_path.exists()
assert figure_path.exists()

metrics_receipt = json.loads(metrics_path.read_text(encoding='utf-8'))
queue = pd.read_csv(queue_path)
print(f"Loaded {len(queue):,} ranked queue rows")
print(metrics_receipt)

Loaded 319,759 ranked queue rows
{'model_family': 'RandomForestClassifier', 'evaluation_split': 'Honest Client-Grouped Split (Unseen Clients)', 'grouped_roc_auc': 0.854, 'grouped_precision_at_50': 0.74, 'grouped_brier_score': 0.117, 'queue_rows_generated': 319759, 'high_priority_actions_share': 0.006}


## 2. Data

The source release is the FlyRank internship warehouse. The analysis joins `dim_content` with the March 2026 partition of `fact_content_daily_performance` at content-item/client grain. Features use March 1–15, 2026 observations; the historical decline proxy compares those impressions with the March 16–31 outcome window.

The model uses impressions, clicks, average position, CTR, active days, and static `word_count`. Client and content IDs are used for joins and grouped validation only. Target-derived fields such as `trend_direction` and `trend_pct`, future-window aggregates, private queries, and client names are excluded.

In [2]:
data_contract = {
    'feature_window': '2026-03-01 through 2026-03-15',
    'outcome_window': '2026-03-16 through 2026-03-31',
    'feature_count': 6,
    'excluded_from_features': ['trend_direction', 'trend_pct', 'client_id', 'content_id']
}
assert data_contract['feature_count'] == 6
assert 'trend_pct' in data_contract['excluded_from_features']
data_contract

{'feature_window': '2026-03-01 through 2026-03-15',
 'outcome_window': '2026-03-16 through 2026-03-31',
 'feature_count': 6,
 'excluded_from_features': ['trend_direction',
  'trend_pct',
  'client_id',
  'content_id']}

## 3. Methodology

A Random Forest classifier ranks pages by estimated decline risk. The transparent Week 4 baseline combines normalized impression volume, click-through deficit, and a thin-content indicator. The primary validation is a `GroupShuffleSplit` by `client_id`, with 39 training clients and 13 unseen test clients; the Week 6 audit also compared this with a naive random split.

The target is an observed proxy: post-cutoff impressions are lower than pre-cutoff impressions. A temporal audit verified that feature rows end on March 15 with zero post-cutoff feature rows. These choices support directional decision assistance, not causal claims about ranking or refresh impact.

In [3]:
validation_receipt = {
    'model_family': metrics_receipt['model_family'],
    'split': metrics_receipt['evaluation_split'],
    'grouped_roc_auc': metrics_receipt['grouped_roc_auc'],
    'grouped_precision_at_50': metrics_receipt['grouped_precision_at_50'],
    'grouped_brier_score': metrics_receipt['grouped_brier_score']
}
assert validation_receipt['split'] == 'Honest Client-Grouped Split (Unseen Clients)'
validation_receipt

{'model_family': 'RandomForestClassifier',
 'split': 'Honest Client-Grouped Split (Unseen Clients)',
 'grouped_roc_auc': 0.854,
 'grouped_precision_at_50': 0.74,
 'grouped_brier_score': 0.117}

## 4. Results (vs baseline)

The model and baseline were evaluated on the same unseen-client split. The model improves the top-50 ranking precision over the heuristic baseline while retaining a calibrated probability score for review prioritization. The naive random split was higher than the grouped estimate, which is why the grouped result is the primary claim.

In [4]:
comparison_table = pd.DataFrame([
    {'System': 'Week-4 heuristic baseline', 'Precision@50': 0.460, 'ROC-AUC': 0.767, 'Brier Score': 0.183},
    {'System': 'Random Forest, client-grouped', 'Precision@50': metrics_receipt['grouped_precision_at_50'], 'ROC-AUC': metrics_receipt['grouped_roc_auc'], 'Brier Score': metrics_receipt['grouped_brier_score']}
])
display(comparison_table)
assert comparison_table.loc[1, 'ROC-AUC'] == validation_receipt['grouped_roc_auc']

,System,Precision@50,ROC-AUC,Brier Score
0,Week-4 heuristic baseline,0.46,0.767,0.183
1,"Random Forest, client-grouped",0.74,0.854,0.117


## 5. Limitations

The label is an observed 15-day impression decline proxy, not a deterministic forecast of future traffic or content quality. Search demand, SERP layouts, tracking changes, and algorithm updates can shift outcomes outside the feature window. Grouped validation measures transfer to held-out clients in this release, but new clients still need a warm-up observation period. Retrospective Precision@50 does not prove that an editorial intervention caused recovery.

In [5]:
limitations = [
    'Historical proxy target, not a deterministic traffic guarantee.',
    'External search and tracking changes are not observed.',
    'New clients require warm-up monitoring.',
    'Retrospective precision does not prove causal editorial impact.'
]
assert len(limitations) == 4
limitations

['Historical proxy target, not a deterministic traffic guarantee.',
 'External search and tracking changes are not observed.',
 'New clients require warm-up monitoring.',
 'Retrospective precision does not prove causal editorial impact.']

## 6. Ranked recommendations

The action playbook translates risk scores into human-review priorities:

| Risk tier | Reason code | Recommended action |
|---|---|---|
| High: $P \ge 0.70$ | `HIGH_IMPRESSION_ZERO_CLICK` | `TITLE_META_CTR_REWRITE`
| High: $P \ge 0.70$ | `THIN_CONTENT_DECAY` | `EXPAND_COMPREHENSIVE_DEPTH`
| Medium: $0.40 \le P < 0.70$ | `RANKING_EROSION` | `INTERNAL_LINKING_AND_CITATIONS`
| Low: $P < 0.40$ | `STABLE_OR_LOW_RISK` | `MONITOR_ROUTINE`

Human reviewers must verify intent, current Search Console data, page type, and recent publishing history. Never automate AI rewrite publication, canonical changes, redirects, URL deletion, or bulk edits to legal, medical, compliance, privacy, terms, branded, checkout, or pricing pages.

In [6]:
action_columns = ['risk_score', 'priority_tier', 'reason_code', 'action_label']
assert set(action_columns).issubset(queue.columns)
assert queue['content_id'].notna().all()
print('Ranked recommendation artifact is present and contains action metadata.')
display(queue[action_columns].head(10))

Ranked recommendation artifact is present and contains action metadata.


,risk_score,priority_tier,reason_code,action_label
0,0.869894,High,CRITICAL_TRAFFIC_COLLAPSE,STRUCTURAL_CONTENT_REFRESH
1,0.866762,High,CRITICAL_TRAFFIC_COLLAPSE,STRUCTURAL_CONTENT_REFRESH
2,0.866727,High,CRITICAL_TRAFFIC_COLLAPSE,STRUCTURAL_CONTENT_REFRESH
3,0.865926,High,CRITICAL_TRAFFIC_COLLAPSE,STRUCTURAL_CONTENT_REFRESH
4,0.865403,High,CRITICAL_TRAFFIC_COLLAPSE,STRUCTURAL_CONTENT_REFRESH
5,0.864381,High,CRITICAL_TRAFFIC_COLLAPSE,STRUCTURAL_CONTENT_REFRESH
6,0.862204,High,CRITICAL_TRAFFIC_COLLAPSE,STRUCTURAL_CONTENT_REFRESH
7,0.860302,High,CRITICAL_TRAFFIC_COLLAPSE,STRUCTURAL_CONTENT_REFRESH
8,0.860092,High,CRITICAL_TRAFFIC_COLLAPSE,STRUCTURAL_CONTENT_REFRESH
9,0.859150,High,CRITICAL_TRAFFIC_COLLAPSE,STRUCTURAL_CONTENT_REFRESH


## 7. Artifacts the paper embeds

The deployed paper embeds the action-distribution figure and links to the metric receipt and executed Week 5–7 notebooks. All paths are relative to the repository and contain anonymized, public-safe outputs.

In [7]:
artifact_receipt = {
    'paper_page': '../index.html',
    'figure': str(figure_path),
    'metrics': str(metrics_path),
    'week_5_model': 'w05_model.ipynb',
    'week_6_validation': 'w06_validation_audit.ipynb',
    'week_7_playbook': 'w07_action_playbook.ipynb'
}
assert all(Path(path).exists() for path in [metrics_path, queue_path, figure_path])
artifact_receipt

{'paper_page': '../index.html',
 'figure': '..\\figures\\playbook_action_distribution.png',
 'metrics': '..\\outputs\\model_metrics.json',
 'week_5_model': 'w05_model.ipynb',
 'week_6_validation': 'w06_validation_audit.ipynb',
 'week_7_playbook': 'w07_action_playbook.ipynb'}

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] The deployed paper, metric receipt, queue, and figure are linked or verified

The capstone summarizes the executed Week 5–7 work; the primary model claim remains the client-grouped result on the historical decline proxy.